In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class HydrazideAcetylation(MorphingOperator):
    def __init__(self):
        super(HydrazideAcetylation, self).__init__()
        self._name = "Hydrazide acetylation"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[NX3;H1][NX3;H2;!$(N-C=O)]")

    def setOriginal(self, mol):
        super(HydrazideAcetylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του ακραίου αζώτου [-NH2] (index 3)
            self._target_atoms.append(match[3])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            terminal_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(terminal_n_idx, c_carbonyl_idx, Chem.BondType.SINGLE)
            
            o_carbonyl_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(c_carbonyl_idx, o_carbonyl_idx, Chem.BondType.DOUBLE)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(c_carbonyl_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
 
            for idx in [terminal_n_idx, c_carbonyl_idx, o_carbonyl_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    

hydrazide_acetylation_op = HydrazideAcetylation()
    

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target


start_mol = MolpherMol("O=C(NN)c1ccncc1") # Ισονιαζίδη
target_mol = MolpherMol("CC(=O)NNC(=O)c1ccncc1") # Ακετυλο-ισονιαζίδη (Στόχος)
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (hydrazide_acetylation_op,)

closest_info = FindClosest()

print("--- STARTING HYDRAZIDE ACETYLATION TREE SEARCH ---")
while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break
print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS: Target reached! The tree completed the Hydrazide Acetylation.")

print("\n=== RUNNING FALSE POSITIVE TRAP TESTS ===")
traps = {
        "Simple Hydrazine Trap (NNc1ccccc1 - Όχι υδραζίδιο, δεν έχει καρβονύλιο)": "NNc1ccccc1",
        "Amide Trap (CC(=O)NC)": "CC(=O)NC",
        "Already Acetylated Trap (CC(=O)NNC(=O)C - Δεν πρέπει να αντιδράσει ξανά)": "CC(=O)NNC(=O)C"
}
    
for name, smiles in traps.items():
    mol = MolpherMol(smiles)
    hydrazide_acetylation_op.setOriginal(mol)
    product = hydrazide_acetylation_op.morph()
        
    safe = (product.getSMILES() == mol.getSMILES())
    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  STATUS: {'SAFE (Passed)' if safe else 'VULNERABLE (Failed)'}")
    print("-" * 50)

--- STARTING HYDRAZIDE ACETYLATION TREE SEARCH ---
Generation #1
Molecules in tree: 2
Closest to target: CC(=O)NNC(=O)C1=CC=NC=C1 (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS: Target reached! The tree completed the Hydrazide Acetylation.

=== RUNNING FALSE POSITIVE TRAP TESTS ===
Simple Hydrazine Trap (NNc1ccccc1 - Όχι υδραζίδιο, δεν έχει καρβονύλιο):
  SOURCE: NNC1=CC=CC=C1
  STATUS: SAFE (Passed)
--------------------------------------------------
Amide Trap (CC(=O)NC):
  SOURCE: CNC(C)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
Already Acetylated Trap (CC(=O)NNC(=O)C - Δεν πρέπει να αντιδράσει ξανά):
  SOURCE: CC(=O)NNC(C)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
